# 第 2 章 · `MemoryProvider`：外部记忆的「插头标准」

对照源码：`../hermes_src/agent/memory_provider.py`

---

## 先搞清楚：这个模块到底干什么？

**一句话**：Hermes 不想把 Mem0 / Honcho / Hindsight 的 SDK 写死在 Agent 里，于是先定了一套 **统一接口**（ABC）。谁想当「外部长期记忆」，就实现这套方法；Agent 只认接口，不认具体厂商。

生活类比：

| 角色 | 类比 |
|------|------|
| `MemoryProvider`（本章） | **USB 插头标准**——规定有哪些针脚 |
| Mem0 / Honcho / … | 某家 U 盘——内部怎么存，Agent 不管 |
| `DemoProvider`（下面演示） | 用纸笔假装的 U 盘——只为演示「插拔流程」 |
| `MemoryManager`（第 3 章） | 电脑上的 USB 口——负责插哪个、一次只插一个外部 |

### 和第 1 章有什么区别？

| | 第 1 章 `MemoryStore` | 第 2 章 `MemoryProvider` |
|--|----------------------|--------------------------|
| 存哪 | 本机 `MEMORY.md` / `USER.md` | 任意外部后端（云 / 向量库 / 图谱…） |
| 谁实现 | Hermes 内置一份 | 插件各写各的 |
| 本章教什么 | 双态 + Prompt Cache | **接口长什么样、何时被调用** |

### ❗ 本章不是在讲什么

- **不是** Mem0 SDK 教程（没有 `mem0.add` / `mem0.search`）
- **不是** 真连云端——`DemoProvider` 只是内存里的一个列表
- 跑完你会看到的是：**Agent 在一轮对话里会按什么顺序叫这些方法**

### 跑完你应该能回答

1. 为什么 Agent 不直接 import Mem0？
2. `system_prompt_block` 和 `prefetch` 差在哪？（静态 vs 动态，关系到 Cache）
3. `sync_turn` / `queue_prefetch` / `prefetch` 分别是「写」「预热」「读」的哪一步？

---

## 接口方法 · 人话版速查

| 方法 | 人话 | 什么时候调 |
|------|------|------------|
| `is_available` | 钥匙配好了吗？（别打网络） | 启动时，决定要不要启用 |
| `initialize` | 开机、连库、建会话 | Session 开始，只一次 |
| `system_prompt_block` | 「我已启用，用法如下」——**整场不变** | 拼 system prompt 时 |
| `prefetch` | 说话前先翻翻旧笔记（要快） | **每一轮** API 调用前 |
| `sync_turn` | 这轮聊完了，记到本子上 | **每一轮**结束后 |
| `queue_prefetch` | 先去后台准备下一轮要用的材料 | 本轮结束后 |
| `get_tool_schemas` | 告诉模型：你可以调哪些记忆工具 | 启动时挂到 tools |
| `handle_tool_call` | 模型真点了「存一条」时执行 | 有 tool_call 时 |
| `on_pre_compress` | 上下文要压缩了，先捞出别丢的事实 | 压缩前 |
| `on_session_end` / `shutdown` | 散会、关门 | 会话结束 |

---

### 生命周期总览

```mermaid
flowchart LR
    A[initialize 开机] --> B[system_prompt_block 静态说明]
    B --> C[prefetch 读旧笔记]
    C --> D[Agent 跟用户聊]
    D --> E[sync_turn 写本子]
    E --> F[queue_prefetch 预热下一轮]
    F --> C
    D -.压缩前.-> G[on_pre_compress]
    D -.散会.-> H[on_session_end / shutdown]
```

### 单轮时序（Agent 眼里长这样）

```mermaid
sequenceDiagram
    participant Agent as AIAgent / MemoryManager
    participant P as MemoryProvider
    participant Model as LLM

    Note over Agent,P: Session 启动（只一次）
    Agent->>P: is_available()
    Agent->>P: initialize(session_id)
    Agent->>P: system_prompt_block()
    Note right of P: 静态说明 → 进 system prompt<br/>整 session 不变 → Cache 友好

    loop 每一轮
        Agent->>P: prefetch(query)
        Note right of P: 动态召回，不进 system prompt
        Agent->>Model: messages + tools
        Model-->>Agent: reply / tool_calls
        opt 模型要存记忆
            Agent->>P: handle_tool_call(...)
        end
        Agent->>P: sync_turn(user, assistant)
        Agent->>P: queue_prefetch(next_query)
    end

    Agent->>P: on_pre_compress / on_session_end / shutdown
```

### 三类通道（别混）

```mermaid
flowchart TD
    subgraph Static[静态 · 启动一次]
        SP[system_prompt_block]
    end
    subgraph Dynamic[动态 · 每轮]
        PF[prefetch]
    end
    subgraph Tools[工具 · 模型主动调]
        HT[handle_tool_call]
    end
    SP -->|写进 system| Cache[Prompt Cache 前缀]
    PF -->|写进本轮上下文| Turn[不破坏 Cache]
    HT -->|显式写入| Store[后端事实库]
```

> **怎么跑**：先 Run ① → 再 Run ② → 最后 Run ③。  
> ③ 的输出按 `STEP 1…6` 分段，每行 `  [方法名]` 就是 Agent 真实会调用的顺序。

## ① 先定义「插头标准」（ABC）

**这一格在干什么？**  
把 Hermes 源码里的 `MemoryProvider` 抄成可执行版本。它本身 **什么也不存、什么也不调**——只规定子类必须有哪些方法。

- **必须实现**：`name` / `is_available` / `initialize` / `get_tool_schemas`
- **有默认空实现、你可覆盖**：`prefetch` / `sync_turn` / `queue_prefetch` / 钩子们
- 跑完只会打印一句 `MemoryProvider ABC 已定义`——正常，标准本身没有演示数据

教学省略了 `on_turn_start` / `on_session_switch` 等边角钩子，主路径够讲清楚即可。

In [ ]:
# =============================================================================
# SOURCE COPY: hermes_src/agent/memory_provider.py :: MemoryProvider
# 这是「标准」本身：没有真实存储。下一格 DemoProvider 才是假实现。
# =============================================================================
from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Any, Dict, List, Optional


class MemoryProvider(ABC):
    """外部记忆后端的抽象基类（插头标准）。

    真系统里：Mem0 / Honcho 各写一个子类。
    MemoryManager（第 3 章）保证同一时刻最多一个外部 Provider。
    """

    # -- 身份 ------------------------------------------------------------------

    @property
    @abstractmethod
    def name(self) -> str:
        """短名字，如 'mem0' / 'honcho'。Manager 用来路由、防重复。"""

    # -- 核心生命周期 ----------------------------------------------------------

    @abstractmethod
    def is_available(self) -> bool:
        """配置/依赖齐了吗？启动时问一次。
        ★ 只查本地配置，不要在这里打网络。"""

    @abstractmethod
    def initialize(self, session_id: str, **kwargs) -> None:
        """会话开机：连后端、暖缓存。每个 session 调一次。"""

    def system_prompt_block(self) -> str:
        """放进 system prompt 的【静态】说明（「我已启用」）。

        ★ 不要把「今天召回的事实」塞这里——每轮一变会打爆 Prompt Cache。
        动态内容请走 prefetch()。
        """
        return ""

    def prefetch(self, query: str, *, session_id: str = "") -> str:
        """每轮说话前：读出相关旧记忆（要快，重活放 queue 里做）。"""
        return ""

    def queue_prefetch(self, query: str, *, session_id: str = "") -> None:
        """本轮结束后：为【下一轮】排队后台召回。默认空操作。"""

    def sync_turn(
        self,
        user_content: str,
        assistant_content: str,
        *,
        session_id: str = "",
        messages: Optional[List[Dict[str, Any]]] = None,
    ) -> None:
        """本轮结束后：把对话写进后端（尽量非阻塞）。"""

    @abstractmethod
    def get_tool_schemas(self) -> List[Dict[str, Any]]:
        """告诉模型：你可以调我哪些工具。没有工具就返回 []。"""

    def handle_tool_call(self, tool_name: str, args: Dict[str, Any], **kwargs) -> str:
        """模型真的点了工具时执行。★ 必须返回 JSON 字符串。"""
        raise NotImplementedError(
            f"Provider {self.name} does not handle tool {tool_name}"
        )

    def shutdown(self) -> None:
        """散会：刷队列、关连接。"""

    # -- 可选钩子 --------------------------------------------------------------

    def on_pre_compress(self, messages: List[Dict[str, Any]]) -> str:
        """上下文要压缩丢掉旧消息前：捞出别丢的事实，塞进 summary。"""
        return ""

    def on_session_end(self, messages: List[Dict[str, Any]]) -> None:
        """会话真正结束时（不是每轮）做汇总抽取。"""


print("✓ MemoryProvider ABC 已定义（这只是标准，还没有任何「记忆」）")

## ② 做一个「假的外部记忆」：`DemoProvider`

**这一格在干什么？**  
实现上一格的全部接口，但后端只是一个 Python 列表 `_facts`——假装是 Mem0 / 向量库。

目的不是教存储算法，而是让你 **能跑通一整条生命周期**，看到每个方法被调用时打印了什么。

| Demo 里的东西 | 真 Provider 里对应 |
|---------------|-------------------|
| `_facts: list` | Mem0 / Honcho 远端库 |
| 关键词匹配 | 向量检索 / 图谱查询 |
| `_queued` | 后台预热线程的缓存 |
| `memory_store_fact` 工具 | 厂商暴露给模型的 memory API |

日志约定：方法里一律 `  [方法名] …`，下一格演示时对着扫即可。

```mermaid
flowchart LR
    subgraph DemoProvider[假后端 DemoProvider]
        Sync[sync_turn 写入] --> Facts[(_facts 列表)]
        Tool[工具 memory_store_fact] --> Facts
        Queue[queue_prefetch] --> Q[_queued]
        Prefetch[prefetch 读取] --> Facts
        Q -.->|优先用排队的 query| Prefetch
    end
```

In [ ]:
import json


def _log(tag: str, msg: str) -> None:
    """教学日志：前缀 [方法名]，方便扫 stdout。"""
    print(f"  [{tag}] {msg}")


class DemoProvider(MemoryProvider):
    """假的外部记忆：用列表当库，用关键词当检索。

    把它当成「Mem0 的纸板模型」——形状一样（同一套方法），里面是假货。
    """

    def __init__(self):
        self._facts: List[str] = []   # 「远端库」——其实就是内存列表
        self._session_id = ""
        self._queued = ""             # 上一轮 queue 下来的查询词
        print("[demo] __init__: 空库就绪（_facts=[]）。这里还没连任何云服务。")

    @property
    def name(self) -> str:
        return "demo"

    def is_available(self) -> bool:
        # 真 Mem0：会查 API Key / SDK 是否安装；教学版永远 True
        _log("is_available", "→ True（教学版跳过凭证检查）")
        return True

    def initialize(self, session_id: str, **kwargs) -> None:
        self._session_id = session_id
        _log("initialize", f"开机 session={session_id!r} kwargs={kwargs or {}}")
        _log("initialize", "真 Provider 这里会：连 API、建 collection、起后台线程")

    def system_prompt_block(self) -> str:
        # ★ 静态说明——整场对话不变，适合进 system prompt（保 Cache）
        block = "[demo provider] Use recalled facts as background only."
        _log("system_prompt_block", "返回【静态】说明（不会每轮变）:")
        _log("system_prompt_block", f"  {block!r}")
        return block

    def prefetch(self, query: str, *, session_id: str = "") -> str:
        # 「读」：优先用上一轮 queue 好的词（模拟后台已预热）
        source = "queued" if self._queued else "arg"
        q = (self._queued or query or "").lower()
        _log("prefetch", f"【读】入参 query={query!r} | 实际检索词来自 {source}={q!r}")
        _log("prefetch", f"当前库: {self._facts}")

        hits = [
            f for f in self._facts
            if any(t in f.lower() for t in q.split() if len(t) > 1)
        ]
        if hits:
            _log("prefetch", f"关键词命中: {hits}")
        else:
            hits = self._facts[-3:]
            _log("prefetch", f"没命中 → 退回最近几条: {hits}")

        if not hits:
            _log("prefetch", "库空 → 不注入")
            return ""
        text = "## Recalled memory\n" + "\n".join(f"- {h}" for h in hits)
        _log("prefetch", f"返回给 Agent 的文本:\n{text}")
        return text

    def queue_prefetch(self, query: str, *, session_id: str = "") -> None:
        # 「预热」：本轮结束记下 query，下一轮 prefetch 直接用
        prev = self._queued
        self._queued = query
        _log("queue_prefetch", f"【预热】_queued: {prev!r} → {query!r}")
        _log("queue_prefetch", "真 Provider：这里起线程打向量库，不阻塞当前回复")

    def sync_turn(self, user_content: str, assistant_content: str, **kwargs) -> None:
        # 「写」：极简规则——用户句里带「我」就当事实收下
        _log("sync_turn", f"【写】user={user_content!r}")
        _log("sync_turn", f"       assistant={assistant_content!r}")
        added = []
        for line in (user_content or "").splitlines():
            s = line.strip()
            keep = bool(s) and ("我" in s or "I " in s) and len(s) < 120 and s not in self._facts
            _log("sync_turn", f"  看句子 {s!r} → {'收下进库' if keep else '跳过'}")
            if keep:
                self._facts.append(s)
                added.append(s)
        _log("sync_turn", f"本轮新增 {len(added)} 条 | 库= {self._facts}")

    def get_tool_schemas(self) -> List[Dict[str, Any]]:
        # 挂给模型的工具：让模型也能「主动存一条」
        schemas = [{
            "name": "memory_store_fact",
            "description": "Persist a durable fact.",
            "parameters": {
                "type": "object",
                "properties": {"fact": {"type": "string"}},
                "required": ["fact"],
            },
        }]
        _log("get_tool_schemas", f"告诉模型可用工具: {[s['name'] for s in schemas]}")
        return schemas

    def handle_tool_call(self, tool_name: str, args: Dict[str, Any], **kwargs) -> str:
        # 模型点了工具 → 显式写入（和 sync_turn 的「自动抽」是两条路）
        _log("handle_tool_call", f"【工具写】tool={tool_name!r} args={args}")
        if tool_name != "memory_store_fact":
            out = json.dumps({"error": f"unknown {tool_name}"})
            _log("handle_tool_call", f"未知工具 → {out}")
            return out
        fact = str(args.get("fact", "")).strip()
        if fact and fact not in self._facts:
            self._facts.append(fact)
            _log("handle_tool_call", f"写入: {fact!r}")
        else:
            _log("handle_tool_call", f"跳过（空或已有）: {fact!r}")
        out = json.dumps(
            {"success": True, "fact": fact, "count": len(self._facts)},
            ensure_ascii=False,
        )
        _log("handle_tool_call", f"返回 JSON（约定）: {out}")
        return out

    def on_pre_compress(self, messages: List[Dict[str, Any]]) -> str:
        _log("on_pre_compress", f"【压缩前】messages={len(messages)} 条将被摘要")
        if not self._facts:
            return ""
        text = "Provider facts to preserve: " + "; ".join(self._facts[-5:])
        _log("on_pre_compress", f"提醒压缩器别丢: {text!r}")
        return text

    def on_session_end(self, messages: List[Dict[str, Any]]) -> None:
        _log("on_session_end", f"【散会】最终事实库= {self._facts}")
        _log("on_session_end", "真 Provider 可在这里做会话级汇总；不是每轮都调")

    def shutdown(self) -> None:
        _log("shutdown", f"【关门】session={self._session_id!r} facts={len(self._facts)}")
        self._queued = ""
        _log("shutdown", "_queued 已清空")


print("✓ DemoProvider 已定义（假后端；下一格会模拟 Agent 按顺序调用它）")

## ③ 模拟 Agent：按真实顺序「插上插头」跑一轮

**这一格在干什么？**  
假装你是 `AIAgent`：按生产环境的顺序调用 `DemoProvider`。  
故事线很简单——用户说「我是 Julie…想学 AI ENGINEER」，系统把它写进库，下一轮再按关键词读出来。

| STEP | 模拟的真实阶段 | 你该看懂的点 |
|------|----------------|--------------|
| 1 | Session 启动 | 静态 SP + tool schema，只做一次 |
| 2 | 一轮聊完 | `sync_turn` **写**；`queue_prefetch` **预热** |
| 3 | 下一轮开口前 | `prefetch` **读**（注意入参 `ignored` 被排队词盖掉） |
| 4 | 模型调工具 | 另一条写入路径（显式存） |
| 5 | 上下文要压缩 | 把事实塞进 summary，防丢失 |
| 6 | 会话结束 | `on_session_end` → `shutdown` |

```mermaid
sequenceDiagram
    participant Demo as DemoProvider
    Note over Demo: STEP1 启动
    Demo->>Demo: sync_turn("我是Julie…")
    Note right of Demo: STEP2 写入 _facts
    Demo->>Demo: queue_prefetch("AI")
    Demo->>Demo: prefetch("ignored")
    Note right of Demo: STEP3 用 queued=AI 读出
    Demo->>Demo: handle_tool_call(...)
    Note right of Demo: STEP4 工具再写一条
    Demo->>Demo: on_pre_compress / end / shutdown
```

In [ ]:
def _banner(step: str, title: str) -> None:
    print("\n" + "=" * 60)
    print(f"  STEP {step} · {title}")
    print("=" * 60)


# ---- 1) Session 启动：只做一次 ---------------------------------------------
_banner("1", "Session 启动（只一次）")
print("人话: 检查能不能用 → 开机 → 拿静态说明 → 挂工具给模型")
p = DemoProvider()
assert p.is_available()
p.initialize("sess-demo", platform="cli", agent_context="primary")
sp = p.system_prompt_block()
schemas = p.get_tool_schemas()
print(f"\n>>> 记住: SP 是静态的，进 system prompt:\n    {sp}")
print(f">>> 记住: 模型能看到的工具名: {[s['name'] for s in schemas]}")

# ---- 2) 一轮聊完：写 + 预热 ------------------------------------------------
_banner("2", "一轮聊完：写入 + 为下一轮预热")
print("人话: 用户自我介绍 → sync_turn 抽进库；再 queue 一个词给下一轮用")
p.sync_turn("我是Julie，我在上海，我想学 AI ENGINEER", "好的，记下了。")
p.queue_prefetch("AI")
print("\n>>> 记住: sync=写后端；queue=还没读，只是预约下一轮的检索词")

# ---- 3) 下一轮开口前：读 ---------------------------------------------------
_banner("3", "下一轮开口前：prefetch 读取")
print("人话: 故意传 query='ignored'，看它是不是优先用排队的 'AI'")
recalled = p.prefetch("ignored")
print(f"\n>>> 记住: 动态召回走本轮上下文，不改 system prompt")
print(f">>> 读到的内容:\n{recalled}")

# ---- 4) 模型主动点工具：另一条写入路径 ------------------------------------
_banner("4", "模型调工具显式存一条")
print("人话: 除了 sync 自动抽，模型还可以主动 memory_store_fact")
tool_result = p.handle_tool_call(
    "memory_store_fact",
    {"fact": "默认时区 Asia/Shanghai"},
)
print(f"\n>>> 记住: tool 返回值必须是 JSON 字符串:\n    {tool_result}")

# ---- 5) 压缩前：别把事实压没 ----------------------------------------------
_banner("5", "上下文要压缩了")
print("人话: 旧消息要被摘要丢掉前，Provider 可以喊一声「这些别丢」")
hint = p.on_pre_compress([])
print(f"\n>>> 塞进 compression summary 的提醒:\n    {hint!r}")

# ---- 6) 散会 --------------------------------------------------------------
_banner("6", "会话结束")
print("人话: 不是每轮都调；真正 /exit 或 session 过期才走这里")
p.on_session_end([])
p.shutdown()

print("\n" + "-" * 60)
print("跑完小结（对照库状态）")
print(f"  _facts  = {p._facts}")
print(f"  _queued = {p._queued!r}  ← shutdown 后应为空")
print("-" * 60)
print("如果你能指着日志说出「写 / 预热 / 读」三步，这一章就过关了。")

## 跑完对照：你刚才模拟了什么？

```text
用户: 我是Julie，我在上海，我想学 AI ENGINEER
        │
        ▼  sync_turn          ← 写进 _facts（自动抽）
        │  queue_prefetch("AI") ← 预约下一轮检索词
        ▼
下一轮: prefetch("ignored")   ← 实际用 "AI" 读出旧事实
        │
        ▼  handle_tool_call   ← 模型再显式存「时区」
        │
        ▼  on_pre_compress / on_session_end / shutdown
```

**模块价值（面试 / 设计时可这么说）**：

1. **解耦**：Agent 核心不绑死 Mem0；换后端只换插件。
2. **Cache 友好**：静态进 system，动态走 prefetch，前缀稳定。
3. **生命周期清晰**：读 / 写 / 预热 / 压缩 / 散会，各有钩子。

**下一章**：`3_memory_manager.ipynb`——谁来注册 Provider、为什么外部同时只允许一个。